# Hito 1 · Recuperación de información
**Guía de supervivencia del veinteañero · Opción A: chatbot documental**

En este hito medimos si dos buscadores léxicos, **TF-IDF y BM25**, encuentran el artículo adecuado para una pregunta. Todavía no generamos respuestas del chatbot.

**Buscamos fragmentos, pero evaluamos documentos.** Aquí un documento es una entrada de `corpus.jsonl` (por ejemplo, un artículo del BOE), no la norma entera.

Guarda este notebook al lado de `README.md` en `PLN_MIAAP`. En VS Code selecciona tu kernel **Python 3.11.9** y ejecuta las celdas de arriba abajo. Las secciones 1 a 6 pueden ejecutarse sin tener aún las preguntas. La sección 7 indica qué falta si todavía no existen.

## 1. Entorno y configuración

Si ya instalaste las dependencias en el kernel que usas, pasa a la primera celda de código. Si faltan, ejecuta `%pip install spacy scikit-learn` en una celda aparte. Para instalar el modelo de español una sola vez, ejecuta `import sys` y, en la línea siguiente, `!"{sys.executable}" -m spacy download es_core_news_sm`. No añadas comillas ni acentos al final.

Las rutas son relativas a la raíz del repositorio. Ejecuta el notebook de arriba abajo desde la carpeta que contiene `data`.

In [43]:
import json
import math
import random
import statistics
from collections import Counter
from pathlib import Path

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = Path("data")
CORPUS = DATA / "corpus.jsonl"
CHUNKS = DATA / "chunks.jsonl"
MUESTRA = DATA / "muestra_consultas.jsonl"
RESPUESTAS = DATA / "respuestas_llm.jsonl"
QUERIES = DATA / "queries.jsonl"

SEMILLA = 42
PALABRAS_POR_FRAGMENTO = 300
SOLAPAMIENTO = 45
CANDIDATOS = 50
MIN_CONSULTAS = 30
K1 = 1.5
B = 0.75


def leer_jsonl(ruta):
    with ruta.open(encoding="utf-8") as archivo:
        return [json.loads(linea) for linea in archivo if linea.strip()]


def guardar_jsonl(ruta, filas):
    with ruta.open("w", encoding="utf-8") as archivo:
        for fila in filas:
            archivo.write(json.dumps(fila, ensure_ascii=False) + "\n")

## 2. Cargar y comprobar el corpus

Leemos el archivo que ya generó el descargador. Comprobamos que los IDs no se repiten, que los textos no están vacíos y qué fuentes se incorporaron realmente. Que exista un HTML en `data/raw` no significa que se hayan extraído sus artículos.

In [44]:
if not CORPUS.is_file():
    raise FileNotFoundError(f"No encuentro {CORPUS.resolve()}. Abre el notebook desde PLN_MIAAP.")

documentos = leer_jsonl(CORPUS)
if not documentos or any(not d["doc_id"] or not d["texto"].strip() for d in documentos):
    raise ValueError("El corpus está vacío o contiene documentos sin ID o sin texto.")

documentos_por_id = {d["doc_id"]: d for d in documentos}
if len(documentos_por_id) != len(documentos):
    raise ValueError("Hay doc_id duplicados en el corpus.")

print("Documentos:", len(documentos))
print("Por fuente:", dict(Counter(d["fuente"] for d in documentos)))
print("Por tema:", dict(Counter(d["tema"] for d in documentos)))

Documentos: 1409
Por fuente: {'BOE': 1409}
Por tema: {'trabajo': 165, 'renta': 226, 'nomina': 170, 'becas': 63, 'ocio': 23, 'consumo': 272, 'conducir': 490}


## 3. Dividir los artículos en fragmentos

La **unidad de indexación** es el fragmento. Los artículos de hasta **300 palabras** quedan en una sola pieza. Los más largos se cortan en ventanas de **300 palabras**, con **45 palabras compartidas** entre ventanas consecutivas (**15 %**). El último fragmento puede ser más corto.

Cada pieza guarda su `chunk_id` propio, el `doc_id` del artículo y las posiciones de inicio y fin. Usamos solo su texto para buscar y conservamos los metadatos del corpus para mostrar los resultados. El archivo `data/chunks.jsonl` se puede reutilizar en el **Hito 2** sin cambiar la segmentación ni los IDs.

In [45]:
def dividir_documento(documento):
    palabras = documento["texto"].split()
    paso = PALABRAS_POR_FRAGMENTO - SOLAPAMIENTO
    piezas = []

    for numero, inicio in enumerate(range(0, len(palabras), paso), start=1):
        fin = min(inicio + PALABRAS_POR_FRAGMENTO, len(palabras))
        piezas.append({
            "chunk_id": f"{documento['doc_id']}_c{numero:04d}",
            "doc_id": documento["doc_id"],
            "texto": " ".join(palabras[inicio:fin]),
            "inicio_palabra": inicio,
            "fin_palabra": fin,
        })
        if fin == len(palabras):
            break

    return piezas


fragmentos = []
for documento in documentos:
    fragmentos.extend(dividir_documento(documento))

fragmentos_por_id = {f["chunk_id"]: f for f in fragmentos}
assert len(fragmentos_por_id) == len(fragmentos), "Hay chunk_id duplicados"
guardar_jsonl(CHUNKS, fragmentos)

longitudes = [f["fin_palabra"] - f["inicio_palabra"] for f in fragmentos]
print("Fragmentos:", len(fragmentos))
print(f"Longitud media: {statistics.mean(longitudes):.1f} palabras")
print("Longitud máxima:", max(longitudes), "palabras")
print("Primer fragmento:", fragmentos[0]["chunk_id"])
print("Guardado en", CHUNKS)

Fragmentos: 2757
Longitud media: 221.1 palabras
Longitud máxima: 300 palabras
Primer fragmento: BOE-A-2015-11430_aunico_c0001
Guardado en data\chunks.jsonl


## 4. Preparar las consultas de evaluación

### 4.1. Seleccionar fragmentos al azar

Tomamos **hasta 50 candidatos** para poder descartar preguntas ambiguas sin quedarnos por debajo de **30 consultas válidas**. La muestra se fija antes de mirar los rankings. Si ya existe, se reutiliza y se comprueba que sus textos e IDs sigan correspondiendo al corpus.

In [46]:
if MUESTRA.exists():
    muestra = leer_jsonl(MUESTRA)
    for candidato in muestra:
        actual = fragmentos_por_id.get(candidato["chunk_id"])
        if actual is None or actual["texto"] != candidato["texto"]:
            raise ValueError("La muestra no coincide con los fragmentos actuales. Revisa el corpus antes de continuar.")
else:
    muestra = random.Random(SEMILLA).sample(fragmentos, min(CANDIDATOS, len(fragmentos)))
    guardar_jsonl(MUESTRA, muestra)

print("Fragmentos seleccionados:", len(muestra))
for candidato in muestra[:2]:
    print("\n", candidato["chunk_id"])
    print(candidato["texto"][:250], "…")

Fragmentos seleccionados: 50

 BOE-A-2009-9481_a4_c0001
Artículo 4. Clases de permiso de conducción y edad requerida para obtenerlo. 1. Todas las clases de permiso de conducción de las que sea titular una persona deberán constar en un único documento con expresión de las categorías de vehículos cuya condu …

 BOE-A-2006-20764_a32_c0001
Artículo 32. Reducciones. 1. Los rendimientos netos con un período de generación superior a dos años, así como aquéllos que se califiquen reglamentariamente como obtenidos de forma notoriamente irregular en el tiempo, se reducirán en un 30 por ciento …


### 4.2. Generar y revisar las preguntas

Pasad al LLM el `chunk_id` y el texto de cada fragmento que queráis usar. Podéis enviarlos por tandas. Usad una instrucción como esta:

> Para cada fragmento, escribe una pregunta natural en español que haría una persona para encontrar esa información. Basa la pregunta solo en el fragmento. **No copies las palabras ni las expresiones distintivas del texto**: reformula con tus propias palabras. No menciones el número del artículo ni el identificador en la pregunta. Devuelve una línea JSON por fragmento, con `chunk_id` y `query`. Conserva el identificador exactamente.

Guardad las respuestas **revisadas** en `data/respuestas_llm.jsonl`, un objeto JSON por línea. **Revisad a mano al menos 10**, comprobando que no copien el texto, que no sean triviales o ambiguas y que el fragmento permita responderlas. Corregid o descartad las que fallen. No hace falta responder a los 50 candidatos si conserváis al menos 30 consultas válidas.

La celda siguiente crea `data/queries.jsonl` con este formato y **no sobrescribe** un conjunto de evaluación existente:

```json
{"qid": "q01", "query": "Pregunta revisada sobre el fragmento", "relevant_ids": ["doc_123"], "source_chunk": "doc_123_c0007"}
```

`relevant_ids` identifica el artículo que sabemos que es relevante. `source_chunk` indica de qué fragmento salió la pregunta. Son etiquetas de evaluación y **no se pasan al buscador**.

#### Cómo documentar la revisión manual

Dejad constancia en el repositorio, por ejemplo en `REVISION_CONSULTAS.md`. Anotad **al menos 10 comprobaciones reales** en una tabla como esta. Usad el `chunk_id` para poder registrar también los descartes, que no tendrán `qid` en el archivo final. No deis por hecha la revisión solo porque las preguntas pasen las comprobaciones del código.

| Fragmento | Consulta revisada | Decisión (aceptar / reformular / descartar) | Motivo y nueva redacción si procede |
|---|---|---|---|
| ID real | Pregunta revisada | Decisión del equipo | ¿Está respaldada por el fragmento? ¿Evita copiar? ¿Es específica y no trivial? |

Tras los descartes, conservad al menos **30 consultas válidas**. Las notas de corrección van en la revisión, no en el campo `query`.


In [47]:
def construir_consultas(respuestas):
    ids_muestra = {f["chunk_id"] for f in muestra}
    preguntas_por_fragmento = {}

    for respuesta in respuestas:
        chunk_id = respuesta["chunk_id"]
        pregunta = respuesta["query"].strip()
        if chunk_id not in ids_muestra:
            raise ValueError(f"Fragmento fuera de la muestra: {chunk_id}")
        if chunk_id in preguntas_por_fragmento or not pregunta:
            raise ValueError(f"Pregunta duplicada por fragmento o vacía: {chunk_id}")
        preguntas_por_fragmento[chunk_id] = pregunta

    consultas = []
    for fragmento in muestra:
        chunk_id = fragmento["chunk_id"]
        if chunk_id in preguntas_por_fragmento:
            consultas.append({
                "qid": f"q{len(consultas) + 1:03d}",
                "query": preguntas_por_fragmento[chunk_id],
                "relevant_ids": [fragmento["doc_id"]],
                "source_chunk": chunk_id,
            })

    return consultas


if QUERIES.exists():
    print("Se conserva el archivo de evaluación existente:", QUERIES)
elif RESPUESTAS.exists():
    nuevas_consultas = construir_consultas(leer_jsonl(RESPUESTAS))
    if len(nuevas_consultas) >= MIN_CONSULTAS:
        guardar_jsonl(QUERIES, nuevas_consultas)
        print("Consultas guardadas:", len(nuevas_consultas), "|", QUERIES)
    else:
        print(f"Hay {len(nuevas_consultas)} consultas. Faltan para llegar a {MIN_CONSULTAS}.")
else:
    print("Pendiente: generar y revisar las preguntas y guardarlas en", RESPUESTAS)

Consultas guardadas: 30 | data\queries.jsonl


## 5. Preprocesar los fragmentos y las consultas con spaCy

1. **Lematizar.** Convertimos cada palabra a su forma base en minúsculas. Por ejemplo, *compré* pasa a *comprar*.
2. **Filtrar por categoría gramatical.** Conservamos nombres, nombres propios, adjetivos, verbos y adverbios. También conservamos números, negaciones y unidades de plazo, aunque spaCy les asigne otra categoría. Quitamos los espacios, la puntuación y las *stopwords* salvo esas excepciones.
3. **Filtrar por frecuencia documental (DF).** Una vez procesados los fragmentos con spaCy, contamos en cuántos aparece cada lema. Por ejemplo, si un lema se repite diez veces en un mismo fragmento, cuenta una sola vez para DF. Como punto de partida, conservamos los lemas que aparecen en **al menos 2 fragmentos** y en **como máximo el 95 %** de los fragmentos. No ajustamos estos umbrales mirando el conjunto de prueba. Son parámetros del notebook, no valores obligatorios del enunciado.

El vocabulario se aprende **solo con los fragmentos** y se aplica después tanto al corpus como a todas las preguntas. Si un número, una negación o un plazo se conserva con spaCy, aún puede quedar fuera por frecuencia documental. No se eliminan fragmentos vacíos tras el filtro. Así se conservan los IDs y el tamaño de la colección.

In [48]:
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
CATEGORIAS = {"NOUN", "PROPN", "ADJ", "VERB", "ADV"}
NEGACIONES = {"no", "ni", "nunca", "jamás", "sin", "tampoco"}
UNIDADES_PLAZO = {"día", "semana", "mes", "año", "hora"}


def extraer_tokens(doc_spacy):
    tokens = []
    for token in doc_spacy:
        if token.is_space or token.is_punct:
            continue

        lema = (token.lemma_ or token.text).lower()
        es_excepcion = (
            token.lower_ in NEGACIONES
            or lema in UNIDADES_PLAZO
            or token.like_num
            or token.pos_ == "NUM"
        )

        if es_excepcion or (token.pos_ in CATEGORIAS and not token.is_stop):
            tokens.append(lema)

    return tokens


pregunta_prueba = "¿Puedo devolver los productos que compré hace 14 días sin recibo?"
print("Tras spaCy, antes de filtrar por DF:", extraer_tokens(nlp(pregunta_prueba)))

Tras spaCy, antes de filtrar por DF: ['devolver', 'producto', 'comprar', '14', 'día', 'sin', 'recibo']


In [49]:
MIN_DF = 2
MAX_DF = 0.95

# Procesamos primero los textos con spaCy.
textos = (fragmento["texto"] for fragmento in fragmentos)
tokens_sin_filtrar_df = []
for doc_spacy in nlp.pipe(textos, batch_size=32):
    tokens_sin_filtrar_df.append(extraer_tokens(doc_spacy))

# Contamos cada lema como máximo una vez por fragmento.
df = Counter()
for tokens in tokens_sin_filtrar_df:
    df.update(set(tokens))

n_fragmentos = len(fragmentos)
vocabulario = set()
for lema, frecuencia in df.items():
    if frecuencia >= MIN_DF and frecuencia / n_fragmentos <= MAX_DF:
        vocabulario.add(lema)

if not vocabulario:
    raise ValueError("El filtro por DF dejó vacío el vocabulario. Revisad los umbrales.")


def filtrar_por_df(tokens):
    return [token for token in tokens if token in vocabulario]


def preprocesar(texto):
    return filtrar_por_df(extraer_tokens(nlp(texto)))


tokens_fragmentos = [filtrar_por_df(tokens) for tokens in tokens_sin_filtrar_df]
print("Fragmentos procesados:", len(tokens_fragmentos))
print(f"Términos antes del filtro DF: {len(df)} | después: {len(vocabulario)}")
print("Fragmentos sin términos tras filtrar:", sum(not tokens for tokens in tokens_fragmentos))
print("Pregunta tras todo el preprocesado:", preprocesar(pregunta_prueba))

Fragmentos procesados: 2757
Términos antes del filtro DF: 9746 | después: 6822
Fragmentos sin términos tras filtrar: 0
Pregunta tras todo el preprocesado: ['devolver', 'producto', 'comprar', '14', 'día', 'sin', 'recibo']


## 6. Construir y probar los dos buscadores

Ambos reciben **los mismos tokens de los mismos fragmentos**. Todos los cálculos de IDF se hacen a nivel de fragmento, no a nivel de artículo completo. La media de longitud para BM25 también se calcula sobre fragmentos, en tokens tras el filtro.

### 6.1. TF-IDF + similitud coseno

Usamos `TfidfVectorizer` de scikit-learn con frecuencia logarítmica (`sublinear_tf=True`) e IDF suavizada. Le pasamos los tokens que ya produjo spaCy, sin que scikit-learn vuelva a tokenizar. Ajustamos el vectorizador **solo con los fragmentos** y usamos ese mismo modelo para transformar preguntas.

In [50]:
def usar_tokens(tokens):
    return tokens


tfidf = TfidfVectorizer(analyzer=usar_tokens, sublinear_tf=True)
matriz_tfidf = tfidf.fit_transform(tokens_fragmentos)


def puntuar_tfidf(tokens_pregunta):
    vector_pregunta = tfidf.transform([tokens_pregunta])
    return cosine_similarity(vector_pregunta, matriz_tfidf)[0]


print("Matriz TF-IDF (fragmentos × términos):", matriz_tfidf.shape)

Matriz TF-IDF (fragmentos × términos): (2757, 6822)


### 6.2. BM25

Implementamos la **variante clásica Okapi BM25 en Python**, sin una biblioteca externa, con `k1 = 1.5`, `b = 0.75` e IDF positiva `log(1 + (N - df + 0.5) / (df + 0.5))`. Aquí `N` es el número de fragmentos y `df` indica en cuántos aparece el término.

Para puntuar cada fragmento recorremos los términos únicos de la pregunta. BM25 reduce el premio por repetir muchas veces un término y ajusta la puntuación según la longitud del fragmento. Dejamos los parámetros fijos, sin ajustarlos con las consultas de evaluación.

In [51]:
frecuencias = [Counter(tokens) for tokens in tokens_fragmentos]
longitudes_tokens = [len(tokens) for tokens in tokens_fragmentos]
longitud_media = statistics.mean(longitudes_tokens)

frecuencia_documental = Counter()
for conteos in frecuencias:
    frecuencia_documental.update(conteos.keys())  # Una vez por término y fragmento.

n_fragmentos = len(fragmentos)
idf_bm25 = {}
for termino, df in frecuencia_documental.items():
    idf_bm25[termino] = math.log1p((n_fragmentos - df + 0.5) / (df + 0.5))


def puntuar_bm25(tokens_pregunta):
    terminos = list(dict.fromkeys(tokens_pregunta))
    puntuaciones = []

    for conteos, longitud in zip(frecuencias, longitudes_tokens):
        ajuste = K1 * (1 - B + B * longitud / longitud_media)
        puntuacion = 0.0

        for termino in terminos:
            frecuencia = conteos[termino]
            if frecuencia > 0:
                puntuacion += idf_bm25[termino] * frecuencia * (K1 + 1) / (frecuencia + ajuste)

        puntuaciones.append(puntuacion)

    return puntuaciones


print(f"BM25: k1={K1}, b={B} | media: {longitud_media:.1f} tokens por fragmento")

BM25: k1=1.5, b=0.75 | media: 107.5 tokens por fragmento


### 6.3. Pasar del ranking de fragmentos al ranking de artículos

Ordenamos todos los fragmentos con puntuación positiva. Si dos empatan, los ordenamos por ID. Como el mejor fragmento de cada artículo aparece primero, **conservar solo esa primera aparición equivale a asignar al artículo la máxima puntuación de sus fragmentos**. Solo después aplicamos el corte K.

In [52]:
METODOS = {"TF-IDF": puntuar_tfidf, "BM25": puntuar_bm25}


def ordenar_fragmentos(puntuaciones):
    candidatos = []
    for fragmento, puntuacion in zip(fragmentos, puntuaciones):
        if puntuacion > 0:
            candidatos.append((fragmento, float(puntuacion)))

    return sorted(candidatos, key=lambda par: (-par[1], par[0]["chunk_id"]))


def agrupar_por_documento(ranking_fragmentos):
    vistos = set()
    ranking_documentos = []

    for fragmento, puntuacion in ranking_fragmentos:
        doc_id = fragmento["doc_id"]
        if doc_id not in vistos:
            vistos.add(doc_id)
            ranking_documentos.append((doc_id, puntuacion))

    return ranking_documentos


def buscar(tokens_pregunta, metodo):
    ranking_fragmentos = ordenar_fragmentos(METODOS[metodo](tokens_pregunta))
    ranking_documentos = agrupar_por_documento(ranking_fragmentos)
    return ranking_fragmentos, ranking_documentos


pregunta_demo = "¿Puedo devolver un producto comprado por internet?"
tokens_demo = preprocesar(pregunta_demo)

print("Consulta de demostración (fuera de la evaluación):", pregunta_demo)
for metodo in METODOS:
    _, resultados = buscar(tokens_demo, metodo)
    print("\n", metodo)
    for puesto, (doc_id, puntuacion) in enumerate(resultados[:3], start=1):
        articulo = documentos_por_id[doc_id]
        print(f"{puesto}. {doc_id} | {articulo.get('seccion', '')} | {puntuacion:.4f}")

Consulta de demostración (fuera de la evaluación): ¿Puedo devolver un producto comprado por internet?

 TF-IDF
1. BOE-A-2007-20555_a6 | Artículo 6 | 0.1576
2. BOE-A-2002-13758_dasegunda | Disposición adicional segunda | 0.1294
3. BOE-A-2007-20555_a75 | Artículo 75 | 0.1283

 BM25
1. BOE-A-2026-10885_a7 | Artículo 7 | 8.9739
2. BOE-A-2007-20555_a114 | Artículo 114 | 8.5572
3. BOE-A-2007-20555_a75 | Artículo 75 | 7.9797


## 7. Evaluar y analizar los errores

### 7.1. Métricas sobre documentos

- **Recall@5 y Recall@10**. Para cada pregunta, dividimos el número de artículos relevantes recuperados entre el número de artículos relevantes conocidos. Con una sola etiqueta por pregunta equivale a un acierto si el artículo aparece dentro de K.
- **MRR**. Calculamos `1 / puesto` del primer artículo relevante y usamos cero si no aparece. Promediamos el resultado sobre todas las preguntas, sin corte para esta métrica.
- **Acierto del fragmento@5 y @10 (opcional)**. Comprobamos si aparece exactamente `source_chunk` en el ranking original de fragmentos, antes de agrupar por artículo.

Las dos primeras métricas de recuperación y el MRR se calculan sobre **artículos únicos**. La tabla solo se genera si hay un archivo de evaluación válido con al menos **30 consultas**.

In [53]:
def validar_consultas(consultas):
    if len(consultas) < MIN_CONSULTAS:
        raise ValueError(f"Se necesitan al menos {MIN_CONSULTAS} consultas válidas.")

    qids = set()
    for consulta in consultas:
        qid = consulta["qid"]
        relevantes = set(consulta["relevant_ids"])
        origen = fragmentos_por_id.get(consulta["source_chunk"])

        if not qid or qid in qids or not consulta["query"].strip():
            raise ValueError(f"Consulta con ID duplicado o campo vacío: {qid}")
        if not relevantes or not relevantes.issubset(documentos_por_id):
            raise ValueError(f"Documentos relevantes no válidos: {qid}")
        if origen is None or origen["doc_id"] not in relevantes:
            raise ValueError(f"El fragmento de origen no pertenece al documento relevante: {qid}")

        qids.add(qid)


def evaluar_consulta(consulta, tokens, metodo):
    ranking_fragmentos, ranking_documentos = buscar(tokens, metodo)
    ids_docs = [doc_id for doc_id, _ in ranking_documentos]
    ids_chunks = [fragmento["chunk_id"] for fragmento, _ in ranking_fragmentos]
    relevantes = set(consulta["relevant_ids"])

    puesto = None
    for posicion, doc_id in enumerate(ids_docs, start=1):
        if doc_id in relevantes:
            puesto = posicion
            break

    fila = {
        "consulta": consulta,
        "puesto": puesto,
        "top5": ids_docs[:5],
        "RR": 1 / puesto if puesto is not None else 0.0,
    }
    for k in (5, 10):
        fila[f"Recall@{k}"] = len(relevantes.intersection(ids_docs[:k])) / len(relevantes)
        fila[f"Fragmento@{k}"] = float(consulta["source_chunk"] in ids_chunks[:k])

    return fila

In [54]:
resultados_por_metodo = {}

if not QUERIES.exists():
    print("Evaluación pendiente. Completa la sección 4 para crear", QUERIES)
else:
    consultas = leer_jsonl(QUERIES)
    validar_consultas(consultas)
    tokens_consultas = [preprocesar(q["query"]) for q in consultas]

    print("Consultas evaluadas:", len(consultas))
    print(f"{'Método':<10} {'Recall@5':>10} {'Recall@10':>10} {'MRR':>8} {'Frag.@5':>9} {'Frag.@10':>9}")

    for metodo in METODOS:
        filas = []
        for consulta, tokens in zip(consultas, tokens_consultas):
            filas.append(evaluar_consulta(consulta, tokens, metodo))
        resultados_por_metodo[metodo] = filas

        claves = ("Recall@5", "Recall@10", "RR", "Fragmento@5", "Fragmento@10")
        medias = [statistics.mean(fila[clave] for fila in filas) for clave in claves]
        print(f"{metodo:<10} {medias[0]:>10.4f} {medias[1]:>10.4f} "
              f"{medias[2]:>8.4f} {medias[3]:>9.4f} {medias[4]:>9.4f}")

Consultas evaluadas: 30
Método       Recall@5  Recall@10      MRR   Frag.@5  Frag.@10
TF-IDF         0.6000     0.6667   0.4044    0.4333    0.6000
BM25           0.6333     0.7333   0.4500    0.5333    0.6000


### 7.2. Analizar los fallos compartidos

Consideramos un fallo compartido en Top 10 cuando **ninguno de los dos métodos** devuelve un documento relevante entre sus diez primeros resultados. Listamos todos los fallos compartidos y mostramos hasta tres en detalle. Si no hay fallos en Top 10, buscamos fallos de ambos en Top 5 y lo indicamos. Si tampoco hay, no inventamos fallos. Como análisis secundario, podéis comparar los puestos en los que difieren los métodos.

Para cada caso, contrastad el fragmento de origen con los primeros resultados. Buscad diferencias de vocabulario entre la consulta y el texto, palabras eliminadas por DF, artículos parecidos o preguntas ambiguas. El puesto y las puntuaciones no prueban la causa del fallo por sí solos. Escribid las conclusiones basadas en lo que observéis.

In [55]:
def fallos_compartidos(k):
    pares = zip(resultados_por_metodo["TF-IDF"], resultados_por_metodo["BM25"])
    fallos = []
    for fila_tf, fila_bm in pares:
        if fila_tf[f"Recall@{k}"] == 0 and fila_bm[f"Recall@{k}"] == 0:
            fallos.append((fila_tf, fila_bm))
    return fallos


if resultados_por_metodo:
    fallos_top10 = fallos_compartidos(10)
    fallos_top5 = fallos_compartidos(5)
    print("Fallos de ambos en Top 10:", len(fallos_top10))
    print("Fallos de ambos en Top 5:", len(fallos_top5))

    fallos = fallos_top10 or fallos_top5
    if not fallos:
        print("No hay fallos compartidos en Top 5 ni en Top 10.")
    else:
        k = 10 if fallos_top10 else 5
        print(f"IDs de los fallos compartidos en Top {k}:")
        print(", ".join(tf["consulta"]["qid"] for tf, _ in fallos))

        for fila_tf, fila_bm in fallos[:3]:
            consulta = fila_tf["consulta"]
            origen = fragmentos_por_id[consulta["source_chunk"]]
            print(f"\n{consulta['qid']} · {consulta['query']}")
            print("Tokens de la pregunta:", preprocesar(consulta["query"]))
            print("Artículos esperados:", ", ".join(consulta["relevant_ids"]))
            print("Fragmento de origen:", origen["chunk_id"])
            print(origen["texto"])

            for metodo, fila in (("TF-IDF", fila_tf), ("BM25", fila_bm)):
                puesto = fila["puesto"] or "no recuperado"
                print(f"  {metodo} | primer relevante: {puesto}")
                for doc_id in fila["top5"][:3]:
                    seccion = documentos_por_id[doc_id].get("seccion", "")
                    print(f"    {doc_id} | {seccion}")
else:
    print("Análisis pendiente hasta ejecutar la evaluación con consultas.")

Fallos de ambos en Top 10: 7
Fallos de ambos en Top 5: 10
IDs de los fallos compartidos en Top 10:
q002, q010, q012, q013, q019, q024, q025

q002 · Soy autónoma y voy a declarar de una vez lo que gané con un encargo puntual que me llevó más de dos años. ¿En qué porcentaje puedo rebajar ese importe en la renta?
Tokens de la pregunta: ['autónomo', 'declarar', 'encargo', 'puntual', 'llevar', 'dos', 'año', 'porcentaje', 'rebajar', 'importe', 'renta']
Artículos esperados: BOE-A-2006-20764_a32
Fragmento de origen: BOE-A-2006-20764_a32_c0001
Artículo 32. Reducciones. 1. Los rendimientos netos con un período de generación superior a dos años, así como aquéllos que se califiquen reglamentariamente como obtenidos de forma notoriamente irregular en el tiempo, se reducirán en un 30 por ciento, cuando, en ambos casos, se imputen en un único período impositivo. La cuantía del rendimiento neto a que se refiere este apartado sobre la que se aplicará la citada reducción no podrá superar el importe de 3

## 8. Antes de entregar

- Conservad `data/chunks.jsonl` y `data/queries.jsonl` para reutilizar los mismos fragmentos y consultas en el Hito 2.
- Ejecutad el notebook completo con las **consultas reales y revisadas** y guardad las salidas visibles.
- En `MILESTONE_1.md`, anotad el tamaño real de la colección, la configuración, Recall@5, Recall@10 y MRR, y comentad dos o tres fallos observados. Indicad el uso del LLM al generar las preguntas y cualquier otra ayuda de IA generativa.
- Incluid o enlazad el registro de la revisión manual de al menos 10 consultas. Documentad los fallos compartidos con ejemplos reales, o indicad que no se observaron.